*   Object-oriented design organizes code into objects that carry data and behavior together.
*   This notebook introduces the Python pieces slowly before mapping them to ML training abstractions.





In [2]:
import math
import random
import numpy as np
import torch

# 3.2.1 Utilities

## 1. Intuition

*   A utility is a small helper that supports the main workflow. It is not the model itself.
*   Object-oriented programming organizes related data and functions into objects.
> *   A class is a template.
> *   An object is one concrete instance made from that template.
> *   An attribute is data stored on an object.
> *   A method is a function attached to an object.
> *   `self` is the conventional name for the object receiving a method call.

## 2. Why this exists

* ML projects repeatedly need helpers for saving settings, tracking metrics, plotting, batching, and training.
* `class` can group these helpers so the code has fewer loose variables.

## 3. Examples

*   Plain Python class: store and update a running average.

In [3]:
class Average: # Same as class Average():

  def __init__(self): # Initiate the total and count elements at 0
    self.total = 0.0
    self.count = 0

  def add(self, value): # Update the running total and count, then return the current average.
    self.total += value
    self.count += 1
    return self.total / self.count

meter = Average()
meter.add(3.0)

3.0

*   Use the same object again, so its attributes remember state.


In [4]:
first = meter.add(5.0)
second = meter.add(7.0)
first, second, meter.total, meter.count # total attribute should be 3+5+7=15, and count should be 1+1+1=3

(4.0, 5.0, 15.0, 3)

## 4. Step-by-step breakdown

*   `class Average:` defines a class template.
*   `__init__` runs when a new object is created.
*   `self.total` and `self.count` are attributes stored on the object.
*   `meter = Average()` creates one object.
*   `meter.add(3.0)` calls the method.
*   Python passes `meter` as `self` automatically.
*   The object remembers previous calls because its attributes remain in memory.

## 5. Connection to ML systems

*   Training code often uses metric accumulators like this to track average loss or accuracy across batches.

## 6. Common confusion points

- `self` is not a keyword, but using that name is standard Python style.
- `__init__` initializes object state; it is not called manually in normal use.
- Methods can change attributes, so method calls may have lasting effects.
- Utility classes should make state clearer, not hide important behavior.

# 3.2.2 Models

## 1. Intuition

*   A model is a function with parameters learned from data.
*   In object-oriented code, a model object usually stores parameters as attributes and provides a method for computing predictions.







## 2. Why this exists

*   Keeping parameters and prediction logic together reduces mistakes.
*   If weights and bias are separate loose variables, it is easier to pass the wrong one into a function.

## 3. Examples

*   Manual model object without PyTorch inheritance.


In [5]:
class LinearModel:

  def __init__(self):
    self.w = torch.tensor([2.0, -1.0])
    self.b = torch.tensor(0.5)

  def predict(self, X):
    return X @ self.w + self.b # Perform X * w + b

model = LinearModel()
model.predict(torch.tensor([1.0, 2.0])) # 2*1 + (-1)*2 = 2 - 2 = 0, then +0.5 = 0.5

tensor(0.5000)

*   PyTorch module version.
*   Inheritance means a class reuses behavior from a parent class.
*   `torch.nn.Module` is the base class for all neural network models in PyTorch. It gives you features like:
> * automatic tracking of parameters
> * saving/loading models
> * moving models to GPU
> * training and evaluation modes
*   Notice how PyTorch randomly initializes the weights and bias.
> * During training, both are updated through backprop and an optimizer (e.g. SGD or Adam) to minimize the loss.




In [6]:
class TorchLinear(torch.nn.Module):

  def __init__(self):
    super().__init__()
    self.layer = torch.nn.Linear(2,1) # Creates a linear layer with 2 inputs and 1 output. Both the weights [w1, w2] and bias are randomly initialized.

  def forward(self, X):
    return self.layer(X)

net = TorchLinear()
net(torch.tensor([1.0, 2.0])) # Computes w1*x1 + w2*x2 + b using the layer's randomly initialized weights and bias, where x1 and x2 are provided by user.

tensor([-0.9585], grad_fn=<ViewBackward0>)

## 4. Step-by-step breakdown

*   `LinearModel` is a plain Python class that stores weights `w` and bias `b`.
> * The input `(X)` is provided when `predict()` is called.
*   `predict` computes the forward prediction formula.
*   `class TorchLinear(torch.nn.Module)` means `TorchLinear` inherits from PyTorch's `Module` class.
*   `super().__init__()` runs the parent class initialization. This lets PyTorch set up internal machinery for parameter tracking.
*   `forward` defines the computation PyTorch should run when the object is called like `net(X)`.











## 5. Connection to ML systems

*   PyTorch models are usually subclasses of `torch.nn.Module`.
*   The parent class manages parameters, nested layers, device movement, and training/evaluation mode.





## 6. Common confusion points

- Inheritance reuses parent behavior; it is not copying code by hand.
- `super().__init__()` is required so PyTorch can initialize module internals.
- `forward` is called indirectly when you write `net(X)`.
- A model object can contain other layer objects as attributes.
> * For example, there are 3 layers in the below model.
> * PyTorch stores each layer (`layer1`, `layer2`, `layer 3`) as an attribute of the model and track their learnable parameters (weights & bias).
   ```
   class BiggerNetwork(torch.nn.Module):

    def __init__(self):
        super().__init__()
        self.layer1 = torch.nn.Linear(10, 20)
        self.layer2 = torch.nn.Linear(20, 5)
        self.layer3 = torch.nn.Linear(5, 1)
   ```



# 3.2.3 Data

## 1. Intuition

*   A data object can store features and labels and provide examples when training code asks for them.
*   An index is a position used to select one item.
*   A dataset often supports `dataset[i]` to return the `i`th example.

## 2. Why this exists

*   Training code should not care whether data came from a CSV file, generated tensors, or images on disk.
*   A dataset object gives the training loop a consistent interface.

## 3. Examples

*   Plain dataset object with `__len__` and `__getitem__`.

In [7]:
class TinyData:

  def __init__(self):
    self.X = torch.tensor([[1.0], [2.0], [3.0]])
    self.y = torch.tensor([2.0, 4.0, 6.0])

  def __len__(self): # "If Python ever sees len(data), it should look for data.__len__() and run it."
    return len(self.y)

  def __getitem__(self, idx): # "When someone indexes a TinyData object, here's how to retrieve the item."
    return self.X[idx], self.y[idx]

data = TinyData()
data[0]

(tensor([1.]), tensor(2.))

*   Create small batches manually from the dataset.

In [8]:
batch_X = torch.stack([data[0][0], data[1][0]])
# data[0] and data[1] call __getitem__(0) and __getitem__(1), returning (X[idx], y[idx]).
# The second [0] selects the first element of that tuple, which is X[idx], giving X[0] and X[1].

batch_y = torch.stack([data[0][1], data[1][1]])
# data[0] and data[1] call __getitem__(0) and __getitem__(1), returning (X[idx], y[idx]).
# The second [1] selects the second element of that tuple, which is y[idx], giving y[0] and y[1].

batch_X, batch_y #X is stacked vertically because X's shape is (3,1) while y's shape is only (3,)

(tensor([[1.],
         [2.]]),
 tensor([2., 4.]))

## 4. Step-by-step breakdown

*   `__len__` is a special method Python calls when you use `len(data)`.
*   `__getitem__` is a special method Python calls when you use square brackets like `data[1]`.
*   The dataset returns one feature tensor and one label tensor.
*   `torch.stack` combines several tensors into a new tensor with a batch dimension.









## 5. Connection to ML systems





*   PyTorch's `Dataset` and `DataLoader` use these same ideas.
> * Our `TinyData` class acts like a Dataset because it provides `__len__()` and `__getitem__()`.
> *  The 2 ssamples we manually created with `torch.stack()` are examples of what a DataLoader produces.
*   A DataLoader repeatedly calls the Dataset's `__getitem__()` method to retrieve samples and combines them into batches.
> * For stacking, DataLoader uses the collation function `default_collate` that combines samples into batches. For normal tensors, this results in stacking behavior.
> * DataLoader also uses `__len__()` when it needs to know the dataset size (to calculate the number of batches).
> * With `DataLoader`, the 2 batch we manually created can be spit out under a single line of code:

    ```
    loader = torch.utils.data.DataLoader(data, batch_size=2)
    ```

    which behaves as follows:

    ```
    Ask dataset:
        give me sample 0
        give me sample 1

    Receive:
        (X[0], y[0])
        (X[1], y[1])

    Stack:
        X[0], X[1] → batch_X
        y[0], y[1] → batch_y
    ```


## 6. Common confusion points

- `__getitem__` receives an index, not a feature value.
- A dataset returns examples; a dataloader returns batches.
- `stack` creates a new dimension; it is different from concatenating along an existing dimension.
- Data abstractions should preserve the pairing between each input and its label.

# 3.2.4 Training

## 1. Intuition

*   Training means repeatedly updating model parameters to reduce loss.
*   An epoch is one pass through the training dataset.
*   A batch is a group of examples taken from the dataset.
> *   During training, the model processes one batch at a time and updates its parameters after each batch.
> *   Batches should be representative of the overall dataset distribution.
> *   Random shuffling helps prevent biased gradient updates and prevents the model from seeing overly similar examples together.
> *   On a large and well-distributed dataset, `DataLoader`'s shuffle and batching behavior is usually sufficient.
> * For imbalanced datasets (e.g., 95% healthy medical scans and 5% disease scans), a custom sampler may be used to balance the batches.
*   An optimizer is the rule that updates parameters using gradients from backprop.
> * Gradient Descent: calculates gradient for the entire dataset$$
w_{t+1}=w_t-\eta\nabla L(w_t)
$$
> * Stochastic Gradient Descent: calculates gradient from a small batch instead of the entire dataset$$
w_{t+1}=w_t-\eta\nabla L(w_t;x_i,y_i)
$$
>> * Take a batch.
>> * Compute predictions.
>> * Compute loss.
>> * Backpropagate to get gradients.
>> * SGD updates the weights.
> * Adam: uses gradients + history of gradients to make smarter updates$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
$$

$$
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
$$

$$
w_{t+1}=w_t-\eta\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}
$$

## Pros and cons of each optimizer

*   Gradient Descent
> * Tries to find the global minimum (or a good enough lower-loss region) error in the entire dataset.
> * Pros:
>> * Very accurate estimate of the true gradient.
>> * Stable/less noisy updates.
> * Cons:
>> * Very slow for large datasets.
>> * Cannot update until you process everything.

*   Stochastic Gradient Descent (SGD)
> *   Let a small batch decide the direction to move to lower-loss region, and compile their opinions together.
> * Pros:
>> * Much faster.
>> * Works well for huge datasets.
>> * Noise can help escape bad solutions.
> * Cons:
>> * Updates are less stable/more noisy.

*   Adam
> *   Keeps exponentially weighted moving averages of past gradients and squared gradients ("Listen to the current batch, but remember what happened before").
> * Pros:
>> * Even faster than SGD.
>> * Is adaptive.
>> * Smoothes over noises with trailing averages.
>> * Requires less tuning of learning rates, momentum, schedules etc.
> * Cons:
>> * Can sometimes generalize worse than SGD on unseen data.
>> * More memory storage given the trailing average.
>> * Less predictable.
>> * Still needs a learning rate.
> * Many LLMs would use AdamW, which combines Adam with decoupled weight decay.
>> * Weight decay encourages smaller weights and can improve generalization.

## 2. Why this exists

*   The training loop is the control center of supervised learning.
*   It defines what runs first, what repeats, what is stored, and when parameters change.





## 3. Examples

*   A minimal training step written directly.

In [14]:
w = torch.tensor([0.0], requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
X = torch.tensor([[1.0], [2.0]])
y = torch.tensor([2.0, 4.0])


# Flatten X from shape (2, 1) to (2,), then compute the predictions:
# y_hat = X * w + b
# Since w = 0 and b = 0, y_hat = [0, 0].
y_hat = X.reshape(-1) * w + b

# Compute the mean squared error (MSE): y_hat - y = [0, 0] - [2, 4] = [-2, -4]
# Square each error: [4, 16]
# Take the mean: (4 + 16) / 2 = 10
loss = ((y_hat - y) ** 2).mean()

# Compute gradients of the loss with respect to all tensors that have
# requires_grad=True (w and b). PyTorch walks backward through the computation graph using the chain rule.
loss.backward()

w.grad, b.grad
# y contains the target values because we used it when defining the loss.
# PyTorch doesn't know y is the "correct answer"—it only differentiates the expression we wrote: ((y_hat - y) ** 2).mean().

# Gradient for w:
  # error = [-2, -4]
  # Multiply each error by its corresponding x value: (-2 * 1) + (-4 * 2) = -10
  # Therefore, w.grad = -10.

# Gradient for b:
  # Sum the errors: (-2) + (-4) = -6
  # Therefore, b.grad = -6.

(tensor([-10.]), tensor(-6.))

*   A trainer-like object can group loop settings.


In [15]:
class TrainerConfig:

  def __init__(self, lr, epochs):
    self.lr = lr
    self.epochs = epochs

config = TrainerConfig(lr=0.03, epochs=3)
# Train the model 3 times at a 0.03 learning rate (weight scale)
# More runs improve the params since the n-1 gradients will update the new runs params closer to the truth (minimum loss location)
# However, too many runs are not good since the learning curve flattens (smaller gradients), and the model memorize (overfit) instead of generalize after repeated exposures
  # A model generalizes when the patterns it learns are useful beyond the training examples.
  # It memorizes when it finds solutions that only work for the specific examples it saw.

config.lr, config.epochs

(0.03, 3)

## 4. Step-by-step breakdown

- The prediction is computed first.

- The loss compares predictions with labels.

- `loss.backward()` computes gradients for `w` and `b`.

- The code stops before updating parameters so the gradient values are visible.

- `TrainerConfig` is not a full trainer. It only shows how training settings can be stored in an object.

## 5. Connection to ML systems

- D2L-style `Trainer` objects eventually package the repeated training workflow.
- Before using them, understand the raw sequence: forward pass, loss, backward pass, parameter update, repeat.

## 6. Common confusion points

- An epoch is not one gradient update unless the dataset has one batch.
> * Instead, gradients update after each batch run, even if you are not yet through the whole dataset.
- Gradients are computed before the optimizer update.
- Training objects should not hide concepts you cannot yet explain manually.
- Framework trainers are convenience layers over ordinary control flow.

# 3.2.5 Summary

## 1. Intuition

*   Object-oriented design groups related state and behavior.
*   For ML, common objects include models, datasets, metric trackers, and trainers.



## 2. Why this exists

*   The point is not to make code look fancy but to reduce accidental complexity as projects grow.


## 3. Examples

*   A tiny object map for a training system.

In [16]:
objects = {
    "model": "stores parameters and predicts",
    "data": "returns examples or batches",
    "trainer": "runs repeated updates",
}
objects


{'model': 'stores parameters and predicts',
 'data': 'returns examples or batches',
 'trainer': 'runs repeated updates'}

## 4. Step-by-step breakdown

* Each object has a responsibility.
> * The model handles prediction.
> * The data object handles example access.
> * The trainer handles repetition and updates.
> * Keeping these responsibilities separate makes later code easier to inspect.

## 5. Connection to ML systems

*   Research code frequently defines custom modules, data objects, and training utilities.
*   Reading them is easier when you can identify each object's responsibility.




## 6. Common confusion points

- Classes are tools for organization, not automatically better code.
- Hidden state can confuse debugging if not named clearly.
- `self` marks object state that persists across method calls.
- Framework abstractions should be mapped back to the manual training loop.

# 3.2.6 Exercises

## 1. Intuition

*   These exercises check whether object-oriented code still feels like ordinary Python control flow.

## 2. Why this exists

*   If a class confuses you, reduce it to attributes, methods, and method-call order.

## 3. Examples

*   Exercise 1: create a metric object and call it twice.


In [19]:
meter = Average()
a = meter.add(2.0)
b = meter.add(6.0)
a, b # 2/1, (2+6)/2

(2.0, 4.0)

*   Exercise 2: inspect a PyTorch module's parameters.

In [20]:
net = TorchLinear()
params = list(net.parameters())
[p.shape for p in params] # For self.w and self.b

[torch.Size([1, 2]), torch.Size([1])]

## 4. Step-by-step breakdown

- Exercise 1 checks persistent object state.

- Exercise 2 checks that PyTorch modules can reveal their trainable parameters.

- The object-oriented interface is useful only if you can still trace what data is stored and what computation runs.

## 5. Connection to ML systems

- Later chapters will rely on module and trainer abstractions.
- These exercises prepare you to read them without treating them as magic.

## 6. Common confusion points

- Calling a method may change object state/modify the object itself.
- `parameters()` comes from PyTorch's parent `Module` class.
- The order of method calls matters in training systems.
- If an abstraction feels unclear, write the manual version beside it.
